# 06 · Mixed precision: autocast, BF16 and GradScaler

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/pytorch/06_mixed_precision.ipynb) &nbsp; 📖 Concept deck: [Mixed precision in PyTorch](https://nilesh-salpe.github.io/visual-ai/pytorch/06-mixed-precision/) &nbsp; ⏱️ Suggested time: **25 min**

**You will build:** what `torch.autocast` changes (matmuls in low precision, reductions kept in FP32), why accumulating in BF16 loses information, and how `GradScaler` scales the loss, skips steps with inf/NaN gradients and adapts its scale.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
print("PyTorch", torch.__version__)

## Task 1: What runs in which dtype?

Inside `torch.autocast("cpu", dtype=torch.bfloat16)`, return a dict of the output dtypes of: `"matmul"` (`a @ b`), `"linear"` (an `nn.Linear` on FP32 input), `"sum"` (`a.sum()`), `"softmax"` (`F.softmax(a, -1)`), and `"param"` (the Linear's weight dtype). Run the code; don't guess.

In [ ]:
def autocast_dtypes():
    a, b = torch.randn(8, 8), torch.randn(8, 8); lin = nn.Linear(8, 8)
    # TODO: with torch.autocast("cpu", dtype=torch.bfloat16): ...
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_autocast_dtypes(f):
    d = f()
    assert d["matmul"] == torch.bfloat16 and d["linear"] == torch.bfloat16, "matmuls run in BF16"
    assert d["sum"] == torch.float32 and d["softmax"] == torch.float32, "reductions and softmax stay in FP32"
    assert d["param"] == torch.float32, "autocast never changes the stored (master) weights"

check("autocast_dtypes", _test_autocast_dtypes, autocast_dtypes)

## Task 2: Why accumulators stay in FP32

Add `x = 1e-3` to a running total 10,000 times, once with the total stored in BF16 and once in FP32. Return `(bf16_total, fp32_total)` as Python floats. BF16 has 8 bits of precision: once the total is large, adding 0.001 rounds away.

In [ ]:
def bf16_accumulate(n=10_000, x=1e-3):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_bf16_accumulate(f):
    b, s = f()
    assert abs(s - 10.0) < 1e-2, "FP32 total ≈ 10"
    assert b < 5.0, f"the BF16 total should stall far below 10, got {b}"

check("bf16_accumulate", _test_bf16_accumulate, bf16_accumulate)

## Task 3: GradScaler skips bad steps and adapts

Run 6 steps of the standard pattern on a tiny model with `scaler = torch.amp.GradScaler("cpu", init_scale=2.**16, growth_interval=2)`: `scaler.scale(loss).backward(); scaler.step(opt); scaler.update()`. On step 2 (0-indexed), multiply the loss by `float("inf")` to simulate an overflow.
Return the list of `scaler.get_scale()` values after each `update()` and whether the weights changed on the overflow step.

In [ ]:
def scaler_steps():
    torch.manual_seed(0); m = nn.Linear(4, 1); opt = torch.optim.SGD(m.parameters(), lr=0.1)
    scaler = torch.amp.GradScaler("cpu", init_scale=2.**16, growth_interval=2)
    scales, changed_on_overflow = [], None
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_scaler_steps(f):
    scales, changed = f()
    assert scales[2] == scales[1] / 2, "an overflow halves the scale (backoff_factor 0.5)"
    assert changed is False, "the optimizer step is skipped when gradients contain inf/NaN"
    assert scales[-1] > scales[2], "after growth_interval clean steps the scale doubles again"

check("scaler_steps", _test_scaler_steps, scaler_steps)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: What runs in which dtype?
def autocast_dtypes_solution():
    a, b = torch.randn(8, 8), torch.randn(8, 8); lin = nn.Linear(8, 8)
    with torch.autocast("cpu", dtype=torch.bfloat16):
        return {"matmul": (a @ b).dtype, "linear": lin(a).dtype, "sum": a.sum().dtype, "softmax": F.softmax(a, -1).dtype, "param": lin.weight.dtype}

check("autocast_dtypes (reference)", _test_autocast_dtypes, autocast_dtypes_solution)

In [ ]:
# @title Solution for Task 2: Why accumulators stay in FP32
def bf16_accumulate_solution(n=10_000, x=1e-3):
    b = torch.zeros((), dtype=torch.bfloat16); s = torch.zeros((), dtype=torch.float32)
    xb, xs = torch.tensor(x, dtype=torch.bfloat16), torch.tensor(x, dtype=torch.float32)
    for _ in range(n):
        b = b + xb; s = s + xs
    return b.item(), s.item()

check("bf16_accumulate (reference)", _test_bf16_accumulate, bf16_accumulate_solution)

In [ ]:
# @title Solution for Task 3: GradScaler skips bad steps and adapts
def scaler_steps_solution():
    torch.manual_seed(0); m = nn.Linear(4, 1); opt = torch.optim.SGD(m.parameters(), lr=0.1)
    scaler = torch.amp.GradScaler("cpu", init_scale=2.**16, growth_interval=2)
    scales, changed = [], None
    for step in range(6):
        opt.zero_grad(); loss = m(torch.randn(8, 4)).pow(2).mean()
        if step == 2: loss = loss * float("inf")
        before = m.weight.detach().clone()
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        if step == 2: changed = not torch.equal(before, m.weight)
        scales.append(scaler.get_scale())
    return scales, changed

check("scaler_steps (reference)", _test_scaler_steps, scaler_steps_solution)

---
## Experiments
**GradScaler defaults** (PyTorch 2.14.1): initial scale 2¹⁶ = 65,536, growth factor 2, backoff factor 0.5, growth interval 2,000 steps. With BF16 you usually don't need a scaler at all; it exists for FP16.

In [ ]:
s = torch.amp.GradScaler("cpu"); print("defaults:", s.get_scale(), s.get_growth_factor(), s.get_backoff_factor(), s.get_growth_interval())
print("autocast default dtype on cpu:", torch.get_autocast_dtype("cpu"), "| on cuda:", torch.get_autocast_dtype("cuda"))
print("BF16 vs FP32 accumulation:", bf16_accumulate_solution())

---
## Takeaways
* `torch.autocast` runs matmuls and linear layers in BF16/FP16 but keeps reductions, softmax and the weights themselves in FP32.
* Long accumulations must stay in FP32; BF16's 8-bit precision silently drops small increments.
* `GradScaler` (for FP16) multiplies the loss, skips steps whose gradients overflow, halves the scale on overflow and doubles it after a run of clean steps.